[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/08_Agua_Superficial/01_agua_superficial.ipynb)

# Agua superficial: ocurrencia, estacionalidad, recurrencia y cambio (JRC)
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [ ]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

In [ ]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "iotapp7"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [ ]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

In [ ]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

In [ ]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

# M8. Agua superficial: ocurrencia, estacionalidad, recurrencia y cambio (JRC)
**Dimensiones:** *ocurrencia* (% del tiempo con agua), *estacionalidad* (nº de meses con agua),
*recurrencia* (% de meses húmedos que repiten agua), *tendencia/cambio* entre épocas.

* **Primario:** `JRC/GSW1_4/GlobalSurfaceWater` + `YearlyHistory` — 30 m, **1984–2021**
  (período nativo fijo del producto: no se puede extender con este dataset; se declara y se
  contrasta con datos recientes si hace falta).
* **Bandas usadas:** `occurrence` (0–100% = **ocurrencia**, no "frecuencia"), `seasonality`
  (0–12 meses), `recurrence` (0–100%), `change_abs` (cambio de ocurrencia entre épocas),
  `YearlyHistory.waterClass` (2 = agua estacional, 3 = agua permanente, por año).
* **Alternativas:** Sentinel-2 MNDWI (para agua post-2021) o `GOOGLE/DYNAMICWORLD`
  (10 m, 2015+, probabilidad de agua) — comentadas en M8.1.

<details>
<summary><b>📐 Corrección de uso (viene del MD guía)</b> — clic para ver</summary>
<br>

En JRC, `occurrence` = **ocurrencia** = % del tiempo que el píxel tuvo agua *en el período
completo* (1984–2021). La **frecuencia** en esta plantilla siempre es un conteo de eventos
(p. ej. nº de años con agua), y la **recurrencia** es % de meses húmedos (jun–sep) en los que
hubo agua. No confundir los tres.

</details>

In [ ]:
# ============================================================
# M8.1 Ocurrencia, estacionalidad y recurrencia del agua
# ============================================================

# --- Primario: JRC GSW 1.4 ----------------------------------------------------
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

ocurrencia = jrc.select("occurrence")        # 0-100 % del tiempo con agua
estacionalidad = jrc.select("seasonality")   # 0-12 meses con agua
recurrencia = jrc.select("recurrence")       # 0-100 % de meses húmedos con agua

# --- Alternativas (comentar primaria y descomentar) ---------------------------
# Agua reciente 10 m (2015+), probabilidad media de agua:
# dw = (
#     ee.ImageCollection("GOOGLE/DYNAMICWORLD")
#     .filterDate("2019-01-01", FECHA_FIN).filterBounds(roi)
#     .select("water").mean()
# )
# ocurrencia = dw.multiply(100).rename("occurrence")   # adaptar el resto de bandas

occ_aoi = promediar_aoi(ocurrencia, "occurrence", ESCALAS["jrc"])
est_aoi = promediar_aoi(estacionalidad, "seasonality", ESCALAS["jrc"])
rec_aoi = promediar_aoi(recurrencia, "recurrence", ESCALAS["jrc"])
print(f"Ocurrencia media en el AOI: {occ_aoi:.1f}% del tiempo hubo agua (1984-2021)")
print(f"Estacionalidad media: {est_aoi:.1f} meses con agua")
print(f"Recurrencia media: {rec_aoi:.1f}% de los meses húmedos repitieron agua")

vis_occ = {"min": 0, "max": 100,
           "palette": ["ffffff", "f7fcf0", "ccebc5", "a8ddb5", "7bccc4", "43a047",
                       "238b45", "006837", "00441b", "00220d"]}
vis_est = {"min": 0, "max": 12,
           "palette": ["ffffff", "edf8fb", "ccece6", "99d8c9", "66c2a4",
                       "2ca25f", "006d2c"]}
vis_rec = {"min": 0, "max": 100,
           "palette": ["ffffff", "f1e3a3", "c7e07d", "7cc26a", "35944c", "0b5c2b"]}

Map_m8 = geemap.Map()
Map_m8.centerObject(roi, ZONA["zoom"])
Map_m8.add_basemap("HYBRID")
Map_m8.addLayer(ocurrencia, vis_occ, "Ocurrencia de agua (%)")
Map_m8.addLayer(estacionalidad, vis_est, "Estacionalidad (meses con agua)", False)
Map_m8.addLayer(recurrencia, vis_rec, "Recurrencia en meses húmedos (%)", False)
Map_m8.addLayer(borde_aoi(), {}, "AOI")
Map_m8.add_colorbar(vis_occ, label="Ocurrencia (% del tiempo con agua)",
                    orientation="horizontal", position="bottomright",
                    layer_name="Ocurrencia (%)")
Map_m8

In [ ]:
# ============================================================
# M8.2 Cambio entre épocas y serie anual de área con agua (1984-2021)
# ============================================================

# --- 1) Cambio de ocurrencia: 2000-2021 vs 1984-1999 (banda change_abs) -------
cambio = jrc.select("change_abs")   # positivo = ganó agua, negativo = perdió agua

cambio_aoi = promediar_aoi(cambio, "change_abs", ESCALAS["jrc"])
print(f"Cambio medio de ocurrencia en el AOI: {cambio_aoi:+.1f} puntos porcentuales "
      "(2000-2021 vs 1984-1999)")

vis_cambio = {"min": -100, "max": 100,
              "palette": ["a50026", "f46d43", "fddbc7", "f7f7f7",
                          "d1e5f0", "4393c3", "053061"]}

# --- 2) Serie anual de área con agua (YearlyHistory, clases 2 y 3) ------------
yh = ee.Image("JRC/GSW1_4/YearlyHistory")
ynombres = yh.bandNames().getInfo()

años_yh = {}
for b in ynombres:
    t = b.lstrip("y")
    if t.isdigit() and len(t) == 4:
        años_yh[int(t)] = b
años_ord = sorted(años_yh)

w_imgs = [
    pix_km2
    .updateMask(yh.select(años_yh[y]).eq(2).Or(yh.select(años_yh[y]).eq(3)))
    .rename(f"w_{y}")
    for y in años_ord
]
d_agua = reducir_bandas(
    w_imgs, [f"w_{y}" for y in años_ord], ESCALAS["jrc"],
    reductor=ee.Reducer.sum(), params=REDUCIR_AREA,
)
df_agua_anio = pd.DataFrame({
    "anio": años_ord,
    "km2_agua": [d_agua.get(f"w_{y}") or 0 for y in años_ord],
})

tend_agua = np.polyfit(df_agua_anio["anio"], df_agua_anio["km2_agua"], 1)[0]
print(f"Serie de agua: {años_ord[0]}-{años_ord[-1]} | "
      f"tendencia {tend_agua:+.3f} km² por año "
      f"({tend_agua * 10:+.2f} km² por década)")

display(df_agua_anio.round(1).tail(10))

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(df_agua_anio["anio"], df_agua_anio["km2_agua"],
        marker="o", ms=4, color="#1976d2", lw=1.4)
ax.set_title("Área con agua (estacional + permanente) por año — JRC YearlyHistory")
ax.set_ylabel("km² con agua")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Map_m8b = geemap.Map()
Map_m8b.centerObject(roi, ZONA["zoom"])
Map_m8b.add_basemap("HYBRID")
Map_m8b.addLayer(cambio, vis_cambio, "Cambio de ocurrencia (puntos %)")
Map_m8b.addLayer(borde_aoi(), {}, "AOI")
Map_m8b.add_colorbar(vis_cambio, label="Δ ocurrencia (2000-21 vs 1984-99)",
                     orientation="horizontal", position="bottomright",
                     layer_name="Cambio de ocurrencia")
Map_m8b

## Interpretación de M8
* **Ocurrencia alta + estacionalidad de 12 meses** = agua permanente (lagos, ríos de régimen
  constante, bofedales con agua); **ocurrencia media + estacionalidad 4–8** = lagunas estacionales
  que aparecen solo en lluvias — recurso clave en temporada seca.
* **Recurrencia** responde *qué tan seguido, entre los meses húmedos, hay agua*: si en jun–sep
  llueve y aun así la recurrencia es baja, el agua superficial es poco confiable.
* **Cambio (zona azul):** ganó agua entre épocas (posible aumento de lagunas o distorsión por
  climáticos húmedos); **zona roja:** perdió agua (secado, extracción, cambio de uso) — priorizar
  verificación en campo.
* **Serie anual:** la pendiente por año resume si el agua superficial crece o se reduce; los años
  de quiebre se comparan con sequías (M5) y con el relato local.
* El producto termina en **2021**; para agua post-2021 usar Sentinel-2/Dynamic World (M8.1).

**Preguntas guía**
1. ¿Las lagunas que ustedes conocen aparecen con estacionalidad alta o baja?
2. ¿Dónde dice el mapa que se perdió agua? ¿coincide con lo observado en campo?
3. ¿Los años más bajos de la serie de área con agua son los mismos de sequía severa (M5)?

## Complemento avanzado: extender el agua mas alla de 2021 con Sentinel-2 MNDWI

El producto JRC termina en **diciembre 2021**. Para agua **reciente** se usa
Sentinel-2 con el indice **MNDWI = (Verde − SWIR1) / (Verde + SWIR1)**.
Metodo heredado del ejercicio avanzado *Poopo / JRC + Sentinel-2* (notebook
`01_long_term_surface_water_jrc.ipynb` del repositorio Transcend): mascara de
nubes con banda `SCL`, compuestos mensuales de mediana, umbral `MNDWI > 0` = agua,
y filtro de cobertura valida ≥ 80 % (el mismo criterio de calidad del JRC mensual).
A diferencia del ejemplo del Poopo (dominio historico del lago), aqui el calculo
corre sobre **`roi` = tu GeoJSON de Bolivia** (el GeoJSON siempre manda).


In [ ]:
# ============================================================
# COMPLEMENTO: serie mensual de agua con Sentinel-2 MNDWI (2019 -> hoy)
# Adaptado del ejercicio Poopo (JRC + S2) al ROI del GeoJSON de Bolivia.
# El GeoJSON (roi) siempre manda: todo se recorta y reduce a `roi`.
# ============================================================
import datetime

S2_START = "2019-01-01"
S2_END = datetime.date.today().replace(day=1).isoformat()  # ultimo mes completo
MIN_VALID_COVERAGE = 80  # % mismo criterio que el JRC mensual

# --- Alternativa de satelite: comentar/descomentar ---
# Primario: COPERNICUS/S2_SR_HARMONIZED (10-20 m, 2017+, MNDWI con B3 verde y B11 SWIR1)
S2_COL = "COPERNICUS/S2_SR_HARMONIZED"
# S2_COL = "COPERNICUS/S2_HARMONIZED"  # TOA (sin correccion atmosferica): mas rapido, menos preciso

def mask_s2(image):
    scl = image.select("SCL")
    clear = (scl.neq(0).And(scl.neq(1)).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    optical = image.select(["B3", "B11"]).multiply(0.0001).updateMask(clear)
    return optical.copyProperties(image, ["system:time_start"])

s2 = (ee.ImageCollection(S2_COL).filterBounds(roi)
      .filterDate(S2_START, S2_END)
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80)).map(mask_s2))
print("Periodo Sentinel-2:", S2_START, "a", S2_END)
print("Imagenes S2:", s2.size().getInfo())

n_months = ee.Date(S2_END).difference(ee.Date(S2_START), "month").round()
offsets = ee.List.sequence(0, n_months.subtract(1))

def monthly_mndwi(offset):
    start = ee.Date(S2_START).advance(offset, "month")
    comp = s2.filterDate(start, start.advance(1, "month")).median()
    mndwi = comp.normalizedDifference(["B3", "B11"]).rename("MNDWI")
    return mndwi.set("date", start.format("YYYY-MM-dd"))

mndwi_monthly = ee.ImageCollection(offsets.map(monthly_mndwi))
print("Meses con compuesto S2:", mndwi_monthly.size().getInfo())

def _area_mes(img):
    m = img.select("MNDWI")
    agua = m.gt(0).unmask(0, False).rename("agua")
    valida = m.mask().unmask(0, False).gt(0).rename("valida")
    pix = ee.Image.pixelArea().divide(1e6).updateMask(m.mask())  # km2 por pixel con dato
    area = pix.addBands(agua.multiply(pix).rename("agua_km2")).addBands(
        valida.multiply(pix).rename("valida_km2"))
    st = area.select(["agua_km2", "valida_km2"]).reduceRegion(
        ee.Reducer.sum(), roi, 20, maxPixels=1e10, tileScale=4, bestEffort=True)
    tot = pix.reduceRegion(ee.Reducer.sum(), roi, 1000,
                           maxPixels=1e9, bestEffort=True).getInfo()
    return ee.Feature(None, {"date": img.get("date"),
                             "agua_km2": st.get("agua_km2"),
                             "valida_km2": st.get("valida_km2"),
                             "aoi_km2": tot.get("area")})

fc = ee.FeatureCollection(mndwi_monthly.map(_area_mes)).getInfo()
recs = [f["properties"] for f in fc["features"]]
df_s2 = pd.DataFrame(recs)
df_s2["date"] = pd.to_datetime(df_s2["date"])
df_s2["cobertura_pct"] = 100 * df_s2["valida_km2"] / df_s2["aoi_km2"]
df_s2["agua_filtrada_km2"] = df_s2["agua_km2"].where(df_s2["cobertura_pct"] >= MIN_VALID_COVERAGE)
print("Meses con cobertura >= 80 %:", int(df_s2["agua_filtrada_km2"].notna().sum()), "de", len(df_s2))
display(df_s2.tail(10))

plt.figure(figsize=(13, 4))
plt.plot(df_s2["date"], df_s2["agua_km2"], color="steelblue", lw=1, alpha=0.35, label="S2 todos los meses")
plt.plot(df_s2["date"], df_s2["agua_filtrada_km2"], color="darkorange", lw=1.5,
         marker="o", ms=3, label="S2 cobertura >= 80 %")
plt.title(f"Agua superficial mensual Sentinel-2 MNDWI sobre el AOI ({S2_START} -> {S2_END})")
plt.ylabel("km2 con agua")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
